# 02 · 트리 베이스라인 V2 (XGBoost / LightGBM, 기본 + 튜닝)

> processed_v2(기존 피처 + 참고 피처 42개)에서 트리를 돌린다.
> 두 가지를 분리해서 본다:
> 1. v2 피처 효과: 기본 파라미터로 v2에서 돌린 결과가 notebooks_v1의 XGBoost 0.2438을 넘나?
> 2. 튜닝 효과: 더 강한 설정(깊이/규제/샘플링)이 PR_AUC를 올리나?
> 기록은 **results_v2.csv** (기존 results.csv와 별개).

## 0. 환경 설정 + 데이터

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import lightgbm as lgb
import xgboost as xgb

from utils import set_seed, compute_metrics, log_result, load_processed
set_seed(42)

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
OUT_DIR      = PROJECT_ROOT / "processed_v2"
NB_DIR       = PROJECT_ROOT / "notebooks_v2"
RESULTS_V2   = NB_DIR / "results_v2.csv"

train_df, val_df, test_df = load_processed(OUT_DIR)
TARGET = "went_on_backorder"
feature_cols = [c for c in train_df.columns if c != TARGET]
X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_val, y_val = val_df[feature_cols], val_df[TARGET]

n_pos = int(y_train.sum()); n_neg = len(y_train) - n_pos
pw = n_neg / n_pos
print("피처:", len(feature_cols), " scale_pos_weight:", round(pw, 1))

피처: 42  scale_pos_weight: 148.5


## 1. XGBoost - 기본 파라미터 (notebooks_v1과 동일 설정)
notebooks_v1의 XGBoost와 같은 설정(300/0.05/depth6)을 v2 피처로 돌린다.
-> 이 값과 notebooks_v1의 0.2438을 비교하면 'v2 피처가 도움이 됐나'를 알 수 있다.

In [2]:
xgb_base = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                             scale_pos_weight=pw, random_state=42, n_jobs=-1, eval_metric="logloss")
xgb_base.fit(X_train, y_train)
m = compute_metrics(y_val, xgb_base.predict_proba(X_val)[:, 1])
log_result("XGB_v2_base", m, path=str(RESULTS_V2))
print("XGB_v2_base", m)

XGB_v2_base {'PR_AUC': 0.2418, 'ROC_AUC': 0.9616, 'Recall': 0.8911, 'Precision': 0.06, 'F1': 0.1124, 'threshold': 0.5}


## 2. XGBoost - 튜닝 설정
더 강한 설정: 트리 수 늘리고(500), 학습률 낮추고(0.03), 깊이 키우고(8),
행/열 샘플링(0.8)과 규제(min_child_weight, reg_lambda, gamma)를 추가해 과적합을 누른다.
(완전한 그리드 서치는 CPU상 비싸서, 대표적인 강한 설정 하나를 쓴다.)

In [3]:
xgb_tuned = xgb.XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=8,
                              subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                              reg_lambda=1.0, gamma=0.1,
                              scale_pos_weight=pw, random_state=42, n_jobs=-1, eval_metric="logloss")
xgb_tuned.fit(X_train, y_train)
xgb_tuned_prob = xgb_tuned.predict_proba(X_val)[:, 1]
m = compute_metrics(y_val, xgb_tuned_prob)
log_result("XGB_v2_tuned", m, path=str(RESULTS_V2))
print("XGB_v2_tuned", m)

XGB_v2_tuned {'PR_AUC': 0.2959, 'ROC_AUC': 0.9687, 'Recall': 0.8699, 'Precision': 0.0861, 'F1': 0.1566, 'threshold': 0.5}


## 3. LightGBM - 기본 / 튜닝

In [4]:
lgb_base = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                              scale_pos_weight=pw, random_state=42, n_jobs=-1, verbose=-1)
lgb_base.fit(X_train, y_train)
m = compute_metrics(y_val, lgb_base.predict_proba(X_val)[:, 1])
log_result("LGBM_v2_base", m, path=str(RESULTS_V2))
print("LGBM_v2_base", m)

lgb_tuned = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=63,
                               subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
                               reg_lambda=1.0, scale_pos_weight=pw, random_state=42, n_jobs=-1, verbose=-1)
lgb_tuned.fit(X_train, y_train)
lgb_tuned_prob = lgb_tuned.predict_proba(X_val)[:, 1]
m = compute_metrics(y_val, lgb_tuned_prob)
log_result("LGBM_v2_tuned", m, path=str(RESULTS_V2))
print("LGBM_v2_tuned", m)

LGBM_v2_base {'PR_AUC': 0.2246, 'ROC_AUC': 0.9568, 'Recall': 0.8831, 'Precision': 0.0551, 'F1': 0.1037, 'threshold': 0.5}


LGBM_v2_tuned {'PR_AUC': 0.2636, 'ROC_AUC': 0.9666, 'Recall': 0.8907, 'Precision': 0.0734, 'F1': 0.1356, 'threshold': 0.5}


## 4. 최고 모델 확률 저장 (04 threshold / 05 ensemble 용)

In [5]:
import numpy as np
# 두 튜닝 모델 중 PR_AUC 높은 쪽을 저장
xgb_ap = compute_metrics(y_val, xgb_tuned_prob)["PR_AUC"]
lgb_ap = compute_metrics(y_val, lgb_tuned_prob)["PR_AUC"]
np.save(NB_DIR / "xgb_v2_val_prob.npy", xgb_tuned_prob)
np.save(NB_DIR / "lgb_v2_val_prob.npy", lgb_tuned_prob)
print("저장: xgb_v2_val_prob.npy, lgb_v2_val_prob.npy")
print("XGB_tuned PR_AUC:", xgb_ap, " LGBM_tuned PR_AUC:", lgb_ap)

저장: xgb_v2_val_prob.npy, lgb_v2_val_prob.npy
XGB_tuned PR_AUC: 0.2959  LGBM_tuned PR_AUC: 0.2636


## 5. 결과 비교 (results_v2.csv)
notebooks_v1 기준선(XGBoost 0.2438)과 비교한다.

In [6]:
res = pd.read_csv(RESULTS_V2).drop_duplicates(subset="model", keep="last").sort_values("PR_AUC", ascending=False)
print("notebooks_v1 기준선: XGBoost(구 피처) PR_AUC 0.2438\n")
res[["model", "PR_AUC", "ROC_AUC", "Recall", "Precision", "F1"]]

notebooks_v1 기준선: XGBoost(구 피처) PR_AUC 0.2438



,model,PR_AUC,ROC_AUC,Recall,Precision,F1
1,XGB_v2_tuned,0.2959,0.9687,0.8699,0.0861,0.1566
3,LGBM_v2_tuned,0.2636,0.9666,0.8907,0.0734,0.1356
0,XGB_v2_base,0.2418,0.9616,0.8911,0.0600,0.1124
2,LGBM_v2_base,0.2246,0.9568,0.8831,0.0551,0.1037


---
### 결론 (실행 후 채움)
- v2 피처 효과: XGB_v2_base vs notebooks_v1 0.2438 비교.
- 튜닝 효과: XGB_v2_tuned vs XGB_v2_base 비교.
- 0.2438을 넘으면 1차 목표 달성. 못 넘으면 04 threshold 최적화로 운영점 개선 + null result 해석.
정형 데이터라 트리가 강한 건 변함없을 가능성이 크다. 숫자로 확인한다.